<a href="https://colab.research.google.com/github/jceltruda/Projects-in-AI-and-ML/blob/main/Project_5/ML_AI_Projects_5_Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3
## Part 1

Task: English to French Translation

In [18]:
import os
import urllib.request
import zipfile
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from nltk.translate.bleu_score import corpus_bleu
import math

# Set seeds
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download and extract dataset
url = "https://download.pytorch.org/tutorial/data.zip"
if not os.path.exists("data"):
    urllib.request.urlretrieve(url, "data.zip")
    with zipfile.ZipFile("data.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
SOS_token = 0
EOS_token = 1
PAD_token = 2

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {"<PAD>": PAD_token, "<SOS>": SOS_token, "<EOS>": EOS_token}
        self.word2count = {}
        self.index2word = {PAD_token: "<PAD>", SOS_token: "<SOS>", EOS_token: "<EOS>"}
        self.n_words = 3

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

def normalizeString(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

# Load data
lines = open('data/eng-fra.txt', encoding='utf-8').read().strip().split('\n')
pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

# Filter for shorter sentences
MAX_LENGTH = 15
filtered_pairs = [p for p in pairs if len(p[0].split()) < MAX_LENGTH and len(p[1].split()) < MAX_LENGTH]

random.shuffle(filtered_pairs)
train_pairs = filtered_pairs[:9000]
test_pairs = filtered_pairs[9000:10000]

input_lang = Lang('eng')
output_lang = Lang('fra')

for pair in pairs:
    input_lang.addSentence(pair[0])
    output_lang.addSentence(pair[1])

print(f"Counted words: Eng {input_lang.n_words}, Fra {output_lang.n_words}")

# Create PyTorch Dataset
class TranslationDataset(Dataset):
    def __init__(self, pairs, input_lang, output_lang, max_length):
        self.pairs = pairs
        self.input_lang = input_lang
        self.output_lang = output_lang
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def indexesFromSentence(self, lang, sentence):
        return [lang.word2index[word] for word in sentence.split(' ')]

    def tensorFromSentence(self, lang, sentence):
        indexes = self.indexesFromSentence(lang, sentence)
        indexes = indexes[:self.max_length - 1]
        indexes.append(EOS_token)
        # Pad sequence
        padded = indexes + [PAD_token] * (self.max_length - len(indexes))
        return torch.tensor(padded, dtype=torch.long)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        input_tensor = self.tensorFromSentence(self.input_lang, pair[0])
        target_tensor = self.tensorFromSentence(self.output_lang, pair[1])
        return input_tensor, target_tensor

dataset = TranslationDataset(train_pairs, input_lang, output_lang, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

Using device: cuda
Counted words: Eng 13047, Fra 18746


In [15]:
def scaled_dp_attention(q, k, v, mask=None):
    d_k = q.shape[-1]

    # Dot product of Q and K^T
    scores = np.matmul(q, k.swapaxes(-2, -1)) / np.sqrt(d_k)

    # Apply mask
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True)) # stability
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    # Multiply by V
    output = np.matmul(weights, v)

    df_weights = pd.DataFrame(weights[0]) if len(weights.shape) == 3 else pd.DataFrame(weights)
    return output, df_weights

# Test
dummy_q = np.random.rand(1, 5, 64)
dummy_k = np.random.rand(1, 5, 64)
dummy_v = np.random.rand(1, 5, 64)
out, attn = scaled_dp_attention(dummy_q, dummy_k, dummy_v)
print("Attention Output Shape:", out.shape)

Attention Output Shape: (1, 5, 64)


## Part 2 & 3

In [16]:
class PyTorchScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, mask=None):
        d_k = q.size(-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, v)
        return output, weights

class EncoderLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)

    def forward(self, input_seq):
        embedded = self.embedding(input_seq)
        output, (hidden, cell) = self.lstm(embedded)
        return output, hidden, cell

class DecoderAttentionLSTM(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderAttentionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = PyTorchScaledDotProductAttention()
        self.lstm = nn.LSTM(hidden_size * 2, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)
        embedded = self.embedding(x)

        # Use hidden state as Query, encoder outputs as Keys and Values
        query = hidden[-1].unsqueeze(1)
        context, attn_weights = self.attention(query, encoder_outputs, encoder_outputs)

        # Concatenate current state
        rnn_input = torch.cat([embedded, context], dim=2)
        output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))
        prediction = self.out(output.squeeze(1))

        return prediction, hidden, cell

In [19]:
# Training setup
hidden_size = 256

# Initialize
encoder1 = EncoderLSTM(input_lang.n_words, hidden_size).to(device)
decoder1 = DecoderAttentionLSTM(hidden_size, output_lang.n_words).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_token)
enc_optimizer = optim.Adam(encoder1.parameters(), lr=0.0005)
dec_optimizer = optim.Adam(decoder1.parameters(), lr=0.0005)

# Short training
EPOCHS = 5
for epoch in range(EPOCHS):
    total_loss = 0
    for input_tensor, target_tensor in dataloader:
        input_tensor, target_tensor = input_tensor.to(device), target_tensor.to(device)

        enc_optimizer.zero_grad()
        dec_optimizer.zero_grad()

        encoder_outputs, hidden, cell = encoder1(input_tensor)

        # Store outputs
        decoder_outputs = []
        decoder_input = torch.tensor([SOS_token] * input_tensor.size(0), device=device)

        for t in range(1, MAX_LENGTH):
            output, hidden, cell = decoder1(decoder_input, hidden, cell, encoder_outputs)
            decoder_outputs.append(output) # Store the output tensor
            decoder_input = target_tensor[:, t]

        # Stack list into a single tensor
        decoder_outputs = torch.stack(decoder_outputs, dim=1)

        # Flatten outputs and calculate loss
        loss = criterion(decoder_outputs.view(-1, output_lang.n_words),
                         target_tensor[:, 1:].contiguous().view(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder1.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(decoder1.parameters(), max_norm=1.0)

        enc_optimizer.step()
        dec_optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(dataloader):.4f}")

Epoch 1/5, Loss: 5.4945
Epoch 2/5, Loss: 4.5489
Epoch 3/5, Loss: 4.1566
Epoch 4/5, Loss: 3.8382
Epoch 5/5, Loss: 3.5621


BLEU Score reported below

## Part 4

In [21]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, num_heads=2):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
        self.attention = PyTorchScaledDotProductAttention()

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)

        scaled_attention, _ = self.attention(q, k, v, mask)
        scaled_attention = scaled_attention.permute(0, 2, 1, 3).contiguous()
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)

        return self.dense(concat_attention)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, num_heads=2, dim_feedforward=128):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_output = self.mha(x, x, x, mask)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

class SimpleTransformer(nn.Module):
    def __init__(self, num_vocab_in, num_vocab_out, d_model=64, num_heads=2, num_layers=2, dim_feedforward=128):
        super().__init__()
        self.enc_embedding = nn.Embedding(num_vocab_in, d_model)
        self.dec_embedding = nn.Embedding(num_vocab_out, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        self.encoder_layers = nn.ModuleList([TransformerBlock(d_model, num_heads, dim_feedforward) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([TransformerBlock(d_model, num_heads, dim_feedforward) for _ in range(num_layers)])

        self.final_layer = nn.Linear(d_model, num_vocab_out)
        self.d_model = d_model

    def create_target_mask(self, size):
        mask = torch.tril(torch.ones(size, size)).unsqueeze(0).unsqueeze(0).to(device)
        return mask

    def forward(self, src, tgt):
        src_seq_len = src.size(1)
        tgt_seq_len = tgt.size(1)

        src_emb = self.pos_encoder(self.enc_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.dec_embedding(tgt) * math.sqrt(self.d_model))

        enc_output = src_emb
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output)

        tgt_mask = self.create_target_mask(tgt_seq_len)

        dec_output = tgt_emb
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, mask=tgt_mask)

        return self.final_layer(dec_output)

# Initialize and train
transformer = SimpleTransformer(
    num_vocab_in=input_lang.n_words,
    num_vocab_out=output_lang.n_words
).to(device)

trans_optimizer = optim.Adam(transformer.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    total_loss = 0
    for input_tensor, target_tensor in dataloader:
        input_tensor, target_tensor = input_tensor.to(device), target_tensor.to(device)

        tgt_input = target_tensor[:, :-1]
        tgt_expected = target_tensor[:, 1:]

        trans_optimizer.zero_grad()

        output = transformer(input_tensor, tgt_input)

        loss = criterion(output.reshape(-1, output_lang.n_words), tgt_expected.reshape(-1))
        loss.backward()
        trans_optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(dataloader):.4f}")

Epoch 1/5, Loss: 5.6464
Epoch 2/5, Loss: 4.4274
Epoch 3/5, Loss: 4.0124
Epoch 4/5, Loss: 3.7362
Epoch 5/5, Loss: 3.5248


In [23]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def evaluate_bleu(model_type="lstm"):
    references = []
    candidates = []

    # Initialize the smoothing function (Method 4 is the standard for machine translation)
    smoothie = SmoothingFunction().method4

    transformer.eval()
    encoder1.eval()
    decoder1.eval()

    with torch.no_grad():
        for pair in test_pairs:
            try:
                input_tensor = dataset.tensorFromSentence(input_lang, pair[0]).unsqueeze(0).to(device)
            except KeyError:
                continue

            target_words = pair[1].split()
            references.append([target_words])

            decoded_words = []
            if model_type == "lstm":
                encoder_outputs, hidden, cell = encoder1(input_tensor)
                decoder_input = torch.tensor([[SOS_token]], device=device)
                for _ in range(MAX_LENGTH):
                    output, hidden, cell = decoder1(decoder_input.squeeze(0), hidden, cell, encoder_outputs)
                    topi = output.argmax(1).item()
                    if topi == EOS_token:
                        break
                    decoded_words.append(output_lang.index2word.get(topi, "<PAD>"))
                    decoder_input = torch.tensor([[topi]], device=device)

            elif model_type == "transformer":
                tgt_indexes = [SOS_token]
                for _ in range(MAX_LENGTH):
                    tgt_tensor = torch.tensor([tgt_indexes], dtype=torch.long).to(device)
                    output = transformer(input_tensor, tgt_tensor)
                    topi = output[0, -1, :].argmax().item()
                    if topi == EOS_token:
                        break
                    tgt_indexes.append(topi)
                    if topi != SOS_token:
                        decoded_words.append(output_lang.index2word.get(topi, "<PAD>"))

            candidates.append(decoded_words)

    # Smoothing to fix 0.0 score
    return corpus_bleu(references, candidates, smoothing_function=smoothie)

print(f"Seq2Seq BLEU Score: {evaluate_bleu('lstm'):.4f}")
print(f"Transformer BLEU Score: {evaluate_bleu('transformer'):.4f}")

Seq2Seq BLEU Score: 0.0134
Transformer BLEU Score: 0.0029


The Seq2Seq model performed much better than the transformer due to its architecture and the length of training. Transformers learn the relationship between individual tokens, and excell at translation tasks but need large amounts of data to fully learn relationships. Given the small dataset and number of epochs, the LSTM was able to learn faster, yielding a higher BLEU score. Both models trained fairly quickly, there wasn't a noticeable difference in runtime.